In [ ]:
# Cell 1 - install/upgrade core libs (latest stable releases)
!pip install -q transformers datasets evaluate accelerate scikit-learn wandb

In [ ]:
# Cell 2 - load dataset
CSV_PATH = "data/query_logs_table.csv"

import pandas as pd
df = pd.read_csv(CSV_PATH)

In [ ]:
# Cell 3 - Cleans dataframe to include only the input/output cols and setup dataset from the dataframe
LABEL_COL   = "likert_1"
DROP_COLS   = {"query_ID", "user_id", "llm_name_1", "query_timestamp"}
TEXT_COLS = [c for c in df.columns if (c not in DROP_COLS) and (c != LABEL_COL)]

SEP = " [SEP] "            # BERT‑friendly separator

# concat text columns into one new column of data
INPUT_COL = "input_text"
df[INPUT_COL] = df[TEXT_COLS].astype(str).agg(SEP.join, axis=1)
# drop all non-relevant columns(keep input_text - inputs and likert_1 - labels)
df = df[[INPUT_COL, LABEL_COL]]

# Normalize the ratings from 1 - 5 to 0 - 4(to match with the model)
# and make sure its a float for regression training
df[LABEL_COL] = (df[LABEL_COL] - 1).astype(float)

from datasets import Dataset
dataset = Dataset.from_pandas(df[["input_text", LABEL_COL]])

df

In [ ]:
# Cell 4 - Tokenize the Dataset
from transformers import AutoTokenizer

# Use a fast and effective model checkpoint
MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# Tokenization function
def tokenize(batch):
  return tokenizer(batch["input_text"], padding=True, truncation=True, max_length=512)

# Apply tokenization to the entire dataset
dataset_encoded = dataset.map(tokenize, batched=True, batch_size=None) # Setting batch_size=None processes the whole dataset at once

# Rename the label column to "labels" for the trainer
dataset_encoded = dataset_encoded.rename_columns({LABEL_COL: "labels"})

In [ ]:
# Cell 5 - Define Metrics and Model
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats

# Performance metrics for a regression task
def compute_metrics_for_regression(eval_pred):
    logits, labels = eval_pred

    # Calculate standard regression metrics
    mse = mean_squared_error(labels, logits)
    mae = mean_absolute_error(labels, logits)
    r2 = r2_score(labels, logits)
    pr = scipy.stats.pearsonr(labels.flatten(), logits.flatten())[0]
    sr = scipy.stats.spearmanr(labels.flatten(), logits.flatten())[0]

    # Calculate custom "accuracy" (predictions within 0.5 of the true label)
    errors = np.abs(logits.flatten() - labels.flatten())
    accuracy = np.sum(errors < 1) / len(errors)

    return {"mse": mse, "mae": mae, "r2": r2, "pearson_r": pr, "spearman_r": sr, "accuracy": accuracy}


# CRITICAL CHANGE: Set num_labels to 1 for regression
num_labels = 1
print(f"Configuring model for regression with {num_labels} output.")

# Load the model with a regression head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=num_labels)

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from transformers import TrainingArguments, Trainer
import gc
import os
import wandb # Import wandb

# --- 1. Setup ---
full_dataset_encoded = dataset_encoded
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
os.environ["WANDB_API_KEY"] = os.environ.get("WANDB_API_KEY", "")

# Give this entire CV experiment a single group name
WANDB_GROUP_NAME = f"CV-bert-base-{pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M')}"

all_fold_metrics = []

# --- 2. Cross-Validation Loop ---
for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset_encoded)):
    print(f"=============== FOLD {fold+1}/{N_SPLITS} ===============")

    # -- A. Create datasets --
    train_fold = full_dataset_encoded.select(train_idx)
    eval_fold = full_dataset_encoded.select(val_idx)

    # -- B. Re-initialize Model --
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=num_labels)

    # -- C. Explicitly initialize W&B for this fold --
    # The trainer will automatically use this active run
    wandb.init(
        project="one-llm-regression-keep-userquery-llmresponse1-adjustment",  # Specify your W&B project name
        group=WANDB_GROUP_NAME,
        name=f"fold-{fold+1}",
        reinit=True, # Important for loops in notebooks
    )

    # -- D. Define Training Arguments --
    output_dir = f"output/{WANDB_GROUP_NAME}/fold-{fold+1}"

    # We no longer need run_name here as wandb.init() handles it
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="mse",
        greater_is_better=False,
        logging_strategy="epoch",
        load_best_model_at_end=True,
        push_to_hub=False,
        # To avoid cluttering the output, we can disable the progress bar for inner loops
    )

    # -- D. Initialize and run the Trainer for this fold --
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=eval_fold,
        compute_metrics=compute_metrics_for_regression,
        tokenizer=tokenizer
    )

    trainer.train()
    metrics = trainer.evaluate()
    all_fold_metrics.append(metrics)
    print(f"Metrics for Fold {fold+1}: {metrics}")

    wandb.finish() # End the W&B run for this fold

    # -- F. Clean up --
    del model
    del trainer
    gc.collect()


# --- 3. Aggregate and Display Final Results ---
print("\n=============== Cross-Validation Complete ===============")
print(f"Metrics were calculated over {N_SPLITS} folds.")

# Create a DataFrame to easily calculate mean and std dev
results_df = pd.DataFrame(all_fold_metrics)

# Display mean and standard deviation for each metric
print("\nAverage Metrics (Mean +/- Std Dev):")
for metric in results_df.columns:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std()
    print(f"- {metric}: {mean_val:.4f} +/- {std_val:.4f}")